## Duplicate Rows

Duplicates are one of the most common data quality issues. They inflate counts, distort averages, and cause joins to fan out unexpectedly. There are two types to watch for:

### Exact duplicates
Every column in the row is identical. These are usually caused by a data loading error (e.g. a file was processed twice) or an overlapping snapshot.

### Duplicates on natural keys
The identifying columns match, but other columns differ. For example, two rows for the same `pupil_id` + `school_urn` but with different `pupil_name` values. This often indicates:
* A data entry error (the same entity recorded inconsistently)
* A legitimate change over time that wasn’t modelled correctly (e.g. a name change without versioning)
* The grain is finer than you assumed (perhaps a date column should be part of the key)

### Strategy for handling duplicates

1. **Detect** — always check before assuming data is clean
2. **Understand** — are they genuine errors or a sign that the grain is wrong?
3. **Decide** — remove exact duplicates with `DISTINCT` or `ROW_NUMBER()`; for key-level duplicates, define a business rule (e.g. keep the latest record)

In [0]:
%sql
-- Recreate the pupil_school view with various duplicate scenarios
CREATE OR REPLACE TEMP VIEW pupil_enrolment AS
SELECT * FROM VALUES
  (1, 'Alice', 100, 'Oak Academy',  '2024-09-01'),
  (2, 'Bob',   100, 'Oak Academy',  '2024-09-01'),
  (3, 'Carol', 200, 'Elm School',   '2024-09-01'),
  (1, 'Alice', 100, 'Oak Academy',  '2024-09-01'),   -- Exact duplicate of row 1
  (2, 'Bob',   100, 'Oak Academy',  '2025-01-06'),   -- Same pupil+school, different date (not exact dup)
  (2, 'Robert',100, 'Oak Academy',  '2025-01-06')    -- Same pupil+school+date, different name!
AS t(pupil_id, pupil_name, school_urn, school_name, enrolment_date);

SELECT * FROM pupil_enrolment;

### Detect exact duplicates

In [0]:
%sql
-- Detect EXACT duplicates: rows where every column is identical
SELECT
  pupil_id, pupil_name, school_urn, school_name, enrolment_date,
  COUNT(*) AS occurrence_count
FROM pupil_enrolment
GROUP BY pupil_id, pupil_name, school_urn, school_name, enrolment_date
HAVING COUNT(*) > 1;

### Detect duplicates on natural keys

In [0]:
%sql
-- Detect duplicates on NATURAL KEY (pupil_id + school_urn)
-- These rows share the same key but differ in other columns
SELECT
  pupil_id,
  school_urn,
  COUNT(*)                        AS row_count,
  COUNT(DISTINCT pupil_name)      AS distinct_names,
  COUNT(DISTINCT enrolment_date)  AS distinct_dates
FROM pupil_enrolment
GROUP BY pupil_id, school_urn
HAVING COUNT(*) > 1;

In [0]:
%sql
-- Remove duplicates using ROW_NUMBER: keep the latest enrolment date per pupil+school
SELECT *
FROM (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY pupil_id, school_urn
      ORDER BY enrolment_date DESC, pupil_name ASC   -- tie-breaker for same date
    ) AS rn
  FROM pupil_enrolment
)
WHERE rn = 1
ORDER BY pupil_id;